In [1]:
"""
Agent 3: Strategic Gap Analysis & Matrix Synthesis Agent (v2 Standalone)
========================================================================
Constructs a Bipartite Material-Reaction Incidence Matrix (M x R) mapping 9 Critical
Mineral Substitution Classes against 6 Energy Reaction Applications. Computes Matrix Sparsity,
identifies Zero-Frequency Research Gaps, and ranks Emerging Low-Density Frontiers.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

class StrategicGapAgent:
    def __init__(self, data_path="agent1_processed_catalysts_3000.xlsx"):
        self.data_path = data_path
        self.df = None
        self.incidence_matrix = None
        self.sparsity = 0.0

    def load_data(self):
        """Loads processed dataset from CSV or Excel across candidate paths."""
        candidate_paths = [
            self.data_path,
            self.data_path.replace(".xlsx", ".csv"),
            "/workspace/artifacts/agent1_processed_catalysts_3000.xlsx",
            "/workspace/scratch/agent1_processed_catalysts_3000.xlsx",
            "/workspace/artifacts/agent1_processed_catalysts_3000.csv",
            "/workspace/scratch/agent1_processed_catalysts_3000.csv"
        ]
        
        found_path = None
        for p in candidate_paths:
            if os.path.exists(p):
                found_path = p
                break
                
        if not found_path:
            raise FileNotFoundError(f"Could not locate dataset in any of: {candidate_paths}")
            
        print(f"[Agent 3] Loading dataset from: {found_path}")
        if found_path.endswith(".csv"):
            self.df = pd.read_csv(found_path)
        else:
            self.df = pd.read_excel(found_path)
            
        print(f"[Agent 3] Ingested {len(self.df)} processed records.")

    def construct_incidence_matrix(self):
        """
        Constructs a Bipartite Incidence Matrix A in R^{9 x 6} mapping
        9 Critical Mineral Material Classes against 6 Energy Reaction Applications.
        """
        cols = self.df.columns
        mat_col = 'extracted_materials' if 'extracted_materials' in cols else ('material_entities' if 'material_entities' in cols else cols[0])
        rxn_col = 'extracted_reactions' if 'extracted_reactions' in cols else ('reaction_type' if 'reaction_type' in cols else cols[1])

        # Cross-tabulate paper counts across Material Classes and Reaction Types
        self.incidence_matrix = pd.crosstab(
            self.df[mat_col].fillna('Unassigned Material'), 
            self.df[rxn_col].fillna('Unassigned Reaction')
        )
        
        # Calculate Matrix Sparsity (AIMA Ch. 6)
        total_cells = self.incidence_matrix.size
        zero_cells = (self.incidence_matrix == 0).sum().sum()
        self.sparsity = (zero_cells / total_cells) * 100.0 if total_cells > 0 else 0.0
        
        print(f"[Agent 3] Constructed Matrix A ({self.incidence_matrix.shape[0]} Materials x {self.incidence_matrix.shape[1]} Reactions).")
        print(f"[Agent 3] Matrix Sparsity: {self.sparsity:.1f}% ({zero_cells}/{total_cells} Zero-Entry Gaps)")

    def rank_strategic_gaps(self):
        """Extracts Zero-Entry Research Gaps and Emerging Low-Density Frontiers (<15 papers)."""
        gap_records = []
        for mat in self.incidence_matrix.index:
            for rxn in self.incidence_matrix.columns:
                count = self.incidence_matrix.loc[mat, rxn]
                if count == 0:
                    status = "Unexplored Zero-Gap"
                    priority = "High Strategic Opportunity"
                elif count < 15:
                    status = "Emerging Low-Density Frontier"
                    priority = "Medium Strategic Opportunity"
                else:
                    status = "Established Research Hub"
                    priority = "Low Strategic Opportunity (Saturated)"
                    
                gap_records.append({
                    'Material_Class': mat,
                    'Energy_Reaction_Application': rxn,
                    'Paper_Count': count,
                    'Research_Status': status,
                    'Strategic_Priority': priority
                })
                
        df_gaps = pd.DataFrame(gap_records).sort_values(by='Paper_Count', ascending=True)
        return df_gaps

    def export_results(self, output_excel="agent3_gap_analysis_results.xlsx"):
        """Exports Matrix Cross-Tabulation and Gap Rankings to an Excel Workbook."""
        df_gaps = self.rank_strategic_gaps()
        
        with pd.ExcelWriter(output_excel) as writer:
            self.incidence_matrix.to_excel(writer, sheet_name="Incidence_Matrix")
            df_gaps.to_excel(writer, sheet_name="Strategic_Gaps_Ranked", index=False)
            
        print(f"[Agent 3] Exported gap analysis results to: {output_excel}")

    def generate_matrix_heatmap(self, output_img="agent3_gap_matrix.png"):
        """Generates publication-quality 9x6 heatmap visualization of the gap matrix."""
        plt.figure(figsize=(11, 7))
        
        sns.heatmap(
            self.incidence_matrix, 
            annot=True, 
            fmt="d", 
            cmap="YlGnBu", 
            linewidths=0.5, 
            linecolor="gray",
            cbar_kws={'label': 'Published Paper Count (3,000 Dataset)'}
        )
        
        plt.title(f"Agent 3: Critical Mineral Replacement vs Energy Reaction Matrix\n(Matrix Sparsity = {self.sparsity:.1f}%)", fontsize=12, fontweight='bold')
        plt.xlabel("Energy Reaction Application", fontsize=10, fontweight='bold')
        plt.ylabel("Critical Mineral Material Class", fontsize=10, fontweight='bold')
        plt.xticks(rotation=30, ha='right')
        plt.yticks(rotation=0)
        plt.tight_layout()
        
        plt.savefig(output_img, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"[Agent 3] Gap matrix heatmap saved to: {output_img}")

    def run_pipeline(self):
        """Executes full Agent 3 pipeline workflow."""
        self.load_data()
        self.construct_incidence_matrix()
        self.export_results()
        self.generate_matrix_heatmap()
        print("[Agent 3] Execution complete!")

if __name__ == "__main__":
    agent = StrategicGapAgent()
    agent.run_pipeline()


[Agent 3] Loading dataset from: agent1_processed_catalysts_3000.xlsx
[Agent 3] Ingested 3000 processed records.
[Agent 3] Constructed Matrix A (9 Materials x 3000 Reactions).
[Agent 3] Matrix Sparsity: 88.9% (24000/27000 Zero-Entry Gaps)
[Agent 3] Exported gap analysis results to: agent3_gap_analysis_results.xlsx
[Agent 3] Gap matrix heatmap saved to: agent3_gap_matrix.png
[Agent 3] Execution complete!
